In [0]:
from pyspark.sql.functions import *

In [0]:
customers_df = spark.table("bronze_customers")

products_df = spark.table("bronze_products")

orders_df = spark.table("bronze_orders")

In [0]:
silver_orders = (
    orders_df.alias("o")
    .join(
        customers_df.alias("c"),
        "customer_id",
        "left"
    )
    .join(
        products_df.alias("p"),
        "product_id",
        "left"
    )
)

In [0]:
silver_orders = (
    silver_orders
    .withColumn(
        "total_amount",
        col("quantity") * col("price")
    )
    .withColumn(
        "order_date",
        to_date("order_date")
    )
)

In [0]:
silver_orders = silver_orders.select(
    "order_id",
    "customer_id",
    "customer_name",
    "city",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "price",
    "total_amount",
    "order_date"
)

In [0]:
silver_orders.show()

In [0]:
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")